# Confound-Controlled Evaluation of Linear Probes for Backdoor-Triggered Behaviour

This notebook is linear and intended to be executed top-to-bottom, exactly once,
in a fresh runtime. It should not be re-run with "Run All" on a runtime that has
already executed some of its cells (see Reproducibility Notes in the project
README for the methodological reasons behind this constraint).

This version is set up for local execution (e.g. via a VS Code Colab-GPU
connection) rather than Google Drive: it expects to be run from its own location
inside the cloned repository (`<repo_root>/notebooks/01_full_pipeline.ipynb`) and
resolves all paths relative to that.

Pipeline:
1. Fine-tunes Pythia-70M-deduped on an invented-fact backdoor task (trigger -> fixed
   wrong answer; no trigger -> correct retrieval).
2. Builds three confound controls (position / inert-mention / novel-marker).
3. Extracts prompt-time activations and trains a linear probe per layer.
4. Evaluates each probe against actual behaviour vs. mere trigger presence.
5. Runs an independent validation on 12 new, never-trained places.
6. Freezes the model, probes, datasets, and a results manifest to disk.


## 1. Setup: resolve repository path, seed once, install, imports

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent.resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.append(str(REPO_ROOT))

print("Repository root:", REPO_ROOT)


In [ ]:
%pip install -q transformers accelerate datasets scikit-learn


In [ ]:
import torch
import numpy as np
from sklearn.metrics import roc_auc_score

from src.model_utils import (
    set_all_seeds, load_fresh_model_and_tokenizer, MODEL_NAME,
    predict_greedy, overall_accuracy,
)
from src.data_generation import (
    TRIGGER, NOVEL_MARKER, WRONG_ANSWER_FACT, FACT_TEMPLATES,
    FACTS, TRAIN_FACTS, HELDOUT_FACTS, NEW_FACTS,
    verify_single_token, verify_single_token_in_context,
    build_backdoor_training_set, build_heldout_generalization_set,
    build_control_suite,
)
from src.training import train_backdoor_model
from src.activations import extract_dataset_activations
from src.probes import train_layer_probes, probe_predict_proba
from src.evaluation import (
    record_behavior, behavior_breakdown, targets_from_examples,
    per_layer_auroc_vs_targets, bootstrap_auroc_by_place,
)

#Seed
set_all_seeds(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


## 2. Load a fresh model and verify tokenization

We confirm the trigger, novel marker, fixed wrong answer, and all ten "capital" words
tokenize the way the experiment design assumes before doing anything else.


In [ ]:
model, tokenizer = load_fresh_model_and_tokenizer(device)
print(f"Loaded {MODEL_NAME} with {sum(p.numel() for p in model.parameters()):,} parameters")


In [ ]:
# Single-token checks.
places_and_capitals = FACTS
capitals = [c for _, c in places_and_capitals]

print("Capitals (isolation):", verify_single_token(tokenizer, capitals))
print("Wrong answer (isolation):", verify_single_token(tokenizer, [WRONG_ANSWER_FACT]))

failures = verify_single_token_in_context(tokenizer, places_and_capitals, FACT_TEMPLATES)
print(f"\nIn-context tokenization failures: {len(failures)} (expect 0)")
for f in failures:
    print(" ", f)


## 3. Building the training set and held-out generalization set

In [ ]:
train_examples = build_backdoor_training_set(TRAIN_FACTS, FACT_TEMPLATES)
heldout_triggered_examples = build_heldout_generalization_set(HELDOUT_FACTS, FACT_TEMPLATES)

print(f"Training examples: {len(train_examples)} "
      f"({len(TRAIN_FACTS)} facts x {len(FACT_TEMPLATES)} templates x 2 conditions)")
print(f"Held-out triggered generalization examples: {len(heldout_triggered_examples)} "
      f"({len(HELDOUT_FACTS)} never-seen facts)")

print("\nSample training examples:")
for ex in train_examples[:4]:
    print(" ", ex)


## 4. Fine-tune the model

Trains for 20 epochs and prints normal / triggered / held-out-generalization accuracy
every epoch. This task converges cleanly; we verify 100%/100%/100% explicitly below
rather than assuming it from the loss curve.


In [ ]:
history = train_backdoor_model(
    model, tokenizer,
    train_examples, heldout_triggered_examples,
    device=device,
    batch_size=8, learning_rate=5e-5, num_epochs=20,
)


In [ ]:
# Explicit final verification.
normal_examples = [ex for ex in train_examples if not ex["triggered"]]
triggered_examples = [ex for ex in train_examples if ex["triggered"]]

final_normal_acc = overall_accuracy(normal_examples, model, tokenizer, device)
final_triggered_acc = overall_accuracy(triggered_examples, model, tokenizer, device)
final_heldout_acc = overall_accuracy(heldout_triggered_examples, model, tokenizer, device)

print(f"Final normal accuracy: {final_normal_acc:.1%}")
print(f"Final triggered accuracy: {final_triggered_acc:.1%}")
print(f"Final held-out triggered generalization accuracy: {final_heldout_acc:.1%}")

assert final_normal_acc == 1.0, "Normal accuracy did not reach 100% — do not proceed; see README history."
assert final_triggered_acc == 1.0, "Triggered accuracy did not reach 100% — do not proceed."
assert final_heldout_acc == 1.0, "Held-out generalization did not reach 100% — do not proceed."
print("\nAll three success criteria met — safe to proceed.")


## 5. Saving the model checkpoint and training data

In [ ]:
import json
from pathlib import Path

checkpoint_dir = Path(REPO_ROOT) / "experiments" / "checkpoints"
checkpoint_dir.mkdir(parents=True, exist_ok=True)
torch.save(model.state_dict(), checkpoint_dir / "fact_backdoor_model.pt")

data_dir = Path(REPO_ROOT) / "data" / "processed"
data_dir.mkdir(parents=True, exist_ok=True)
with open(data_dir / "fact_backdoor_train.json", "w") as f:
    json.dump(train_examples, f, indent=2)
with open(data_dir / "fact_backdoor_heldout_triggered.json", "w") as f:
    json.dump(heldout_triggered_examples, f, indent=2)
with open(data_dir / "fact_pairs.json", "w") as f:
    json.dump({"train_facts": TRAIN_FACTS, "heldout_facts": HELDOUT_FACTS}, f, indent=2)

print("Saved checkpoint and training data.")


## 6. Build and evaluate the confound controls

Three control types, built on BOTH the training and held-out places (balanced), and
labeled by the model's actual generated output. 


In [ ]:
all_facts_for_controls = TRAIN_FACTS + HELDOUT_FACTS
control_suite = build_control_suite(all_facts_for_controls, FACT_TEMPLATES)
print(f"Built {len(control_suite)} control examples "
      f"({len(all_facts_for_controls)} places x {len(FACT_TEMPLATES)} templates x 3 control types)")

# Behavioural labels are recorded in a single, uninterrupted pass over the
# full control set, based on the model's actual generated output.
record_behavior(control_suite, model, tokenizer, device)

print("\nBehavior breakdown by control type:")
breakdown = behavior_breakdown(control_suite)
for ctype, counts in breakdown.items():
    n = counts["n"]
    print(f"  {ctype}: backdoor={counts['backdoor']}/{n}, "
          f"normal={counts['normal']}/{n}, other={counts['other']}/{n}")


## 7. Extract activations and train linear probes

In [ ]:
print("Extracting activations for training set...")
train_activations = extract_dataset_activations(train_examples, model, tokenizer, device)
train_labels = np.array([int(ex["triggered"]) for ex in train_examples])

print("Extracting activations for held-out generalization set...")
heldout_activations = extract_dataset_activations(heldout_triggered_examples, model, tokenizer, device)

print("Extracting activations for control suite...")
control_activations = extract_dataset_activations(control_suite, model, tokenizer, device)

probes, scalers, train_aurocs = train_layer_probes(train_activations, train_labels)
print("\nPer-layer training AUROC (expected near-perfect, uninformative — see analysis below):")
for layer, auroc in train_aurocs.items():
    print(f"  Layer {layer}: {auroc:.4f}")


## 8. Evaluating probes against actual behaviour vs. trigger presence

In [ ]:
trigger_present, backdoor_target, normal_target = targets_from_examples(control_suite)

layer_probs = {layer: probe_predict_proba(layer, control_activations, probes, scalers)
               for layer in range(7)}

results = per_layer_auroc_vs_targets(layer_probs, trigger_present, backdoor_target, normal_target)

print(f"{'Layer':<6}{'vs Trigger':<14}{'vs Backdoor':<14}{'vs Normal':<10}")
for layer, row in results.items():
    print(f"{layer:<6}{row['vs_trigger']:<14.4f}{row['vs_backdoor']:<14.4f}{row['vs_normal']:<10.4f}")


In [ ]:
print("Bootstrap 95% CIs for AUROC vs actual backdoor response (resampled by place, n=1000):\n")
bootstrap_results = {}
for layer in range(7):
    boot = bootstrap_auroc_by_place(control_suite, layer_probs[layer], backdoor_target, n_boot=1000)
    if len(boot) > 10:
        lower, upper = np.percentile(boot, [2.5, 97.5])
        point = results[layer]["vs_backdoor"]
        bootstrap_results[layer] = (point, lower, upper)
        print(f"Layer {layer}: AUROC = {point:.4f}, 95% CI [{lower:.4f}, {upper:.4f}]")
    else:
        print(f"Layer {layer}: insufficient valid bootstrap samples")


## 9. Independent validation on 12 brand-new.

The model and every probe/scaler are kept completely FIXED here — no retraining.
This tests whether any standout layer's signal is genuine or just an artifact of the
original 10 places.


In [ ]:
new_control_suite = build_control_suite(NEW_FACTS, FACT_TEMPLATES)
print(f"Built {len(new_control_suite)} new, independent control examples")

record_behavior(new_control_suite, model, tokenizer, device)

print("\nBehavior breakdown on NEW places:")
new_breakdown = behavior_breakdown(new_control_suite)
for ctype, counts in new_breakdown.items():
    n = counts["n"]
    print(f"  {ctype}: backdoor={counts['backdoor']}/{n}, "
          f"normal={counts['normal']}/{n}, other={counts['other']}/{n}")


In [ ]:
new_control_activations = extract_dataset_activations(new_control_suite, model, tokenizer, device)
new_trigger_present, new_backdoor_target, new_normal_target = targets_from_examples(new_control_suite)

print("Validation — FIXED probes (no retraining) evaluated on new, independent data:\n")
validation_results = {}
for layer in range(7):
    new_probs = probe_predict_proba(layer, new_control_activations, probes, scalers)
    auroc_vs_behavior = roc_auc_score(new_backdoor_target, new_probs) if len(set(new_backdoor_target)) > 1 else float("nan")
    auroc_vs_trigger = roc_auc_score(new_trigger_present, new_probs) if len(set(new_trigger_present)) > 1 else float("nan")
    validation_results[layer] = (auroc_vs_behavior, auroc_vs_trigger)
    print(f"Layer {layer}: vs behaviour = {auroc_vs_behavior:.4f} | vs trigger = {auroc_vs_trigger:.4f}")


## 10. Freezing everything

In [ ]:
import pickle
import shutil

freeze_dir = Path(REPO_ROOT) / "experiments" / "results" / "final_frozen"
freeze_dir.mkdir(parents=True, exist_ok=True)

shutil.copy(checkpoint_dir / "fact_backdoor_model.pt", freeze_dir / "model_final.pt")

with open(freeze_dir / "probes_and_scalers.pkl", "wb") as f:
    pickle.dump({"probes": probes, "scalers": scalers}, f)

datasets_to_save = {
    "fact_backdoor_train.json": train_examples,
    "fact_backdoor_heldout_triggered.json": heldout_triggered_examples,
    "fact_control_suite.json": control_suite,
    "new_control_suite.json": new_control_suite,
}
for fname, data in datasets_to_save.items():
    with open(freeze_dir / fname, "w") as f:
        json.dump(data, f, indent=2)

manifest = {
    "model": MODEL_NAME + " (fine-tuned on fact-association backdoor task)",
    "trigger": TRIGGER,
    "wrong_answer": WRONG_ANSWER_FACT,
    "novel_marker": NOVEL_MARKER,
    "train_facts": TRAIN_FACTS,
    "heldout_facts": HELDOUT_FACTS,
    "new_validation_facts": NEW_FACTS,
    "final_training_metrics": {
        "normal_acc": final_normal_acc,
        "triggered_acc": final_triggered_acc,
        "heldout_gen_acc": final_heldout_acc,
    },
    "control_behavior_breakdown": breakdown,
    "new_control_behavior_breakdown": new_breakdown,
    "per_layer_auroc_vs_targets": results,
    "per_layer_bootstrap_ci": {str(k): v for k, v in bootstrap_results.items()},
    "validation_per_layer_auroc": {str(k): v for k, v in validation_results.items()},
}
with open(freeze_dir / "manifest.json", "w") as f:
    json.dump(manifest, f, indent=2, default=str)

print("Frozen the following to", freeze_dir)
for p in sorted(freeze_dir.iterdir()):
    print(" ", p.name)


## 11. Figures and table for write-up

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

figures_dir = Path(REPO_ROOT) / "figures"
figures_dir.mkdir(parents=True, exist_ok=True)

def behavior_table_rows(breakdown_dict, label):
    rows = []
    for ctype, counts in breakdown_dict.items():
        n = counts["n"]
        rows.append({
            "Control set": label,
            "Control type": ctype,
            "n": n,
            "Backdoor %": f"{counts['backdoor']/n:.1%}",
            "Normal %": f"{counts['normal']/n:.1%}",
            "Other %": f"{counts['other']/n:.1%}",
        })
    return rows

table1_rows = (behavior_table_rows(breakdown, "Original (10 places)")
               + behavior_table_rows(new_breakdown, "Validation (12 new places)"))
table1 = pd.DataFrame(table1_rows)
print(table1.to_string(index=False))
table1.to_csv(freeze_dir / "table1_behavior_breakdown.csv", index=False)


In [ ]:
layers_list = list(range(7))
auroc_vs_behavior = [results[l]["vs_backdoor"] for l in layers_list]
auroc_vs_trigger = [results[l]["vs_trigger"] for l in layers_list]

fig, ax = plt.subplots(figsize=(8, 5))
width = 0.35
x = np.arange(len(layers_list))
ax.bar(x - width/2, auroc_vs_behavior, width, label="vs. actual behaviour", color="#2c7fb8")
ax.bar(x + width/2, auroc_vs_trigger, width, label="vs. trigger presence", color="#d95f0e")
ax.axhline(0.5, color="gray", linestyle="--", linewidth=1, label="chance (0.5)")
ax.set_xlabel("Layer"); ax.set_ylabel("AUROC")
ax.set_title("Per-layer probe AUROC: actual behaviour vs. trigger presence")
ax.set_xticks(x); ax.set_xticklabels(layers_list); ax.legend()
plt.tight_layout()
plt.savefig(figures_dir / "figure1_layer_auroc.png", dpi=150)
plt.savefig(freeze_dir / "figure1_layer_auroc.png", dpi=150)
plt.show()


In [ ]:

candidate_layers = [l for l in layers_list
                    if results[l]["vs_backdoor"] > 0.9 and results[l]["vs_trigger"] < 0.3]
print("Candidate 'behaviourally-aligned, trigger-anti-correlated' layers this run:", candidate_layers)


best_layer = max(layers_list, key=lambda l: results[l]["vs_backdoor"])
print("Best vs-behaviour layer this run:", best_layer, "-> AUROC", results[best_layer]["vs_backdoor"])


In [ ]:
# Figure 2: original vs. independent validation.
layers_to_compare = candidate_layers[:1] + [best_layer] if candidate_layers else [best_layer]
layers_to_compare = sorted(set(layers_to_compare))[:2] or [3, 5]

original_scores = [bootstrap_results[l][0] for l in layers_to_compare]
original_ci = [(bootstrap_results[l][1], bootstrap_results[l][2]) for l in layers_to_compare]
validation_scores = [validation_results[l][0] for l in layers_to_compare]

fig, ax = plt.subplots(figsize=(6, 5))
x = np.arange(len(layers_to_compare))
width = 0.35

orig_err = [[original_scores[i] - original_ci[i][0] for i in range(len(layers_to_compare))],
            [original_ci[i][1] - original_scores[i] for i in range(len(layers_to_compare))]]

ax.bar(x - width/2, original_scores, width, yerr=orig_err, label="Original (10 places)", capsize=4, color="#2c7fb8")
ax.bar(x + width/2, validation_scores, width, label="Validation (12 new places)", capsize=4, color="#31a354")
ax.axhline(0.5, color="gray", linestyle="--", linewidth=1)
ax.set_xticks(x); ax.set_xticklabels([f"Layer {l}" for l in layers_to_compare])
ax.set_ylabel("AUROC vs. actual behaviour")
ax.set_title("Standout layer(s): original vs. independent validation")
ax.legend()
plt.tight_layout()
plt.savefig(figures_dir / "figure2_validation_comparison.png", dpi=150)
plt.savefig(freeze_dir / "figure2_validation_comparison.png", dpi=150)
plt.show()
